# Darcy with user-written primal Galerkin equations

The user supplies four blocks; the library eliminates the local complement:

$$
\begin{aligned}
A_T u_T+B_T\lambda_T &= f_T,\\
C_Tu_T+D_T\lambda_T &= g_T.
\end{aligned}
$$

Here $a_T(p,v)=(\nabla p,\nabla v)_T$, $L_T(v)=(-4,v)_T$,
$B_T\lambda=\langle\lambda,v\rangle_{\partial T}$ and
$C_Tp=-\langle p,\mu\rangle_{\partial T}$ with signed macroface normals.
$D_T=0$ and $g_T=0$. The constant local kernel is retained globally;
the physical integral selects its complement. The global right-hand side is
$-\langle p_D,\mu\rangle_{\partial\Omega}$ in trace coordinates, followed
by zeros in retained-mode coordinates. No model name chooses these forms.

The pressure is $p=1+x^2+y^2$ and the raw Darcy flux is $q=-2(x,y)$. The explicit P2 application cell verifies both fields.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/7dedb99de6d0e6108351b4d7815b834a7a32da61e9ade9610429bdad0791764c/primal_galerkin-companion.zip"
COMPANION_SHA256 = "7dedb99de6d0e6108351b4d7815b834a7a32da61e9ade9610429bdad0791764c"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/primal_galerkin.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


In [ ]:
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from pymhm.fem.scalar.triangle import scalar_operators, trace_coupling
from examples.tutorial_local_provider import (
    exact_pressure, exact_flux, source, TutorialResult, diagnostics,
)

mesh = TriangleMesh.unit_square()
skeleton = SkeletonSpace(mesh)
local_degree, local_refinement = 2, 2

def local_equations(cell):
    """Write local pressure equations and the independent global balance rows."""
    fine = mesh.submesh(cell, local_refinement)
    a, mass, L = scalar_operators(
        fine, local_degree, diffusion=1.0, source=source, order=4,
    )
    b = trace_coupling(mesh, cell, fine, skeleton, local_degree)
    kernel = np.ones((a.shape[0], 1))
    physical_mean = mass @ kernel
    return LocalEquations(
        a, L, columns(*b.T), rows(*(-b.T)), skeleton.cell_dofs(cell),
        kernel=kernel, moments=physical_mean / (kernel.T @ physical_mean).item(),
        metadata=(fine, physical_mean[:, 0]),
    )

boundary, fixed = boundary_data(skeleton, exact_pressure, order=4)
global_equation = Equation(0, -np.r_[boundary, np.zeros(len(mesh.cells))])
problem = MultiscaleProblem(
    global_equation, local_equations, range(len(mesh.cells)), skeleton.size,
    (1,) * len(mesh.cells), fixed=fixed,
)
system = assemble(problem)
solution = system.solve()
result = TutorialResult(problem, system, solution, "primal", "dirichlet", "basix")
record = diagnostics(result)
print(record)
assert max(record["field_L2_errors"].values()) < 1e-10


## Change geometry, source or boundary convention

The shared patch functions keep exact data, moments and integration choices
identical to the package's regression checks. Every record declares its local
and trace spaces. `primal-quadratic` uses $p=1+x^2+2y^2$ and $f=-6$.
`primal-neumann` prescribes outward physical flux and mean pressure $2.5$.
The 3D option uses $p=1+x+2y+3z$. These are small analytical checks.

The comparison below selects declared configurations from the application catalogue. Its drivers assemble the public mathematical providers in [companion formulation providers](https://github.com/ipes-lncc/pymhm/tree/main/examples/formulations/) through the same generic local/global API. Each configuration records its approximation spaces and boundary conventions.


In [ ]:
from examples.tutorial_scalar_variants import run_variant

selected_methods = ('primal', 'primal-quadratic', 'primal-neumann', 'primal-rectangle', 'primal3d')
records = [run_variant(name) for name in selected_methods]
for record in records:
    print(record["variant"], record["spaces_and_conventions"], record["metrics"])
